# Native Language Identification using HuBERT Features - GPU Version

**GPU-Accelerated Training for Deep Learning Models**

This notebook is optimized for GPU training on Google Colab or systems with CUDA support.

## What's Included
- Automatic CUDA detection and GPU memory management
- Fast HuBERT feature extraction using GPU
- GPU-accelerated deep learning training (CNN, BiLSTM, Transformer)
- Mixed precision training for faster convergence
- Batch processing for large datasets

## Prerequisites
- Google Colab with GPU runtime (T4/V100/A100)
- OR local machine with CUDA-capable GPU

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/NLI_HuBERT_Project/

In [ ]:
# Check GPU availability and specs
import torch
print("="*70)
print("GPU ENVIRONMENT CHECK")
print("="*70)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Device Count: {torch.cuda.device_count()}")
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected. Will fall back to CPU.")
print("="*70)

In [ ]:
#  FAST HUGGINGFACE HUBERT EXTRACTOR (optimized for NLI_HuBERT_Project)
# Resume-safe, GPU accelerated, batched

import os, torch, numpy as np, librosa
from tqdm.auto import tqdm
from transformers import Wav2Vec2FeatureExtractor, HubertModel

# CONFIG
WAV_DIR = "/content/drive/MyDrive/NLI_HuBERT_Project/data/raw"
EMB_DIR = "/content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy"
MODEL_NAME = "facebook/hubert-base-ls960"
BATCH_SIZE = 12            # Adjust: T4=8–12, V100/A100=24–32
SR = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True             # Mixed precision = faster on GPU


print(f"🚀 Device: {DEVICE} | Using AMP: {USE_AMP}")
os.makedirs(EMB_DIR, exist_ok=True)

# Load model + feature extractor
extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = HubertModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

# Collect all wav files recursively
wav_files = []
for root, _, files in os.walk(WAV_DIR):
    for f in files:
        if f.endswith(".wav"):
            wav_files.append(os.path.join(root, f))
print("🎧 Total WAV files found:", len(wav_files))

# Skip already extracted
existing = {os.path.splitext(f)[0] for f in os.listdir(EMB_DIR)}
to_process = [p for p in wav_files if os.path.splitext(os.path.basename(p))[0] not in existing]
print(f"✅ Already extracted: {len(existing)} | Remaining: {len(to_process)}")

# Process in batches
for i in tqdm(range(0, len(to_process), BATCH_SIZE), desc="HuBERT Extraction (batched)"):
    batch_paths = to_process[i:i+BATCH_SIZE]
    audios, ids = [], []
    for p in batch_paths:
        try:
            y, _ = librosa.load(p, sr=SR)
            audios.append(y)
            ids.append(os.path.splitext(os.path.basename(p))[0])
        except Exception as e:
            print("⚠️ Load failed:", p, e)
    if not audios:
        continue

    inputs = extractor(audios, sampling_rate=SR, return_tensors="pt", padding=True)
    with torch.no_grad():
        if USE_AMP and DEVICE == "cuda":
            with torch.cuda.amp.autocast():
                out = model(inputs["input_values"].to(DEVICE))
        else:
            out = model(inputs["input_values"].to(DEVICE))
        embs = out.last_hidden_state.mean(dim=1).cpu().numpy()

    for idx, emb in zip(ids, embs):
        np.save(os.path.join(EMB_DIR, f"{idx}.npy"), emb.astype(np.float32))

print(f"\n✅ Extraction complete! Saved embeddings to: {EMB_DIR}")

In [ ]:
 for test_bs in [8, 12, 16, 24, 32]:
    try:
        torch.cuda.empty_cache()
        dummy = torch.randn(test_bs, 16000 * 5, device="cuda")  # simulate 5s clips
        print(f"✅ {test_bs} fits in memory")
    except RuntimeError:
        print(f"❌ OOM at batch {test_bs}")
        break

In [ ]:
from pathlib import Path
import numpy as np

BASE = Path("/content/drive/MyDrive/NLI_HuBERT_Project")
HUBERT_OLD = BASE / "features" / "hubert"         # old .npz (540 files)
HUBERT_NEW = BASE / "features" / "hubert_npy"     # new .npy
WAV_DIR = BASE / "data" / "raw"

print("📂 Project Base:", BASE)
print("📂 WAV Directory Exists:", WAV_DIR.exists())
print("📂 Old HuBERT (.npz):", HUBERT_OLD.exists())
print("📂 New HuBERT (.npy):", HUBERT_NEW.exists())

# Count files
old_count = sum(1 for _ in HUBERT_OLD.rglob("*.npz")) if HUBERT_OLD.exists() else 0
new_count = sum(1 for _ in HUBERT_NEW.rglob("*.npy")) if HUBERT_NEW.exists() else 0

print(f"\n📊 Old extractor (.npz): {old_count} files")
print(f"📊 New extractor (.npy): {new_count} files")

# List a few samples
if new_count > 0:
    print("\n🔍 First 10 extracted .npy files:")
    for i, p in enumerate(sorted(HUBERT_NEW.rglob("*.npy"))):
        if i >= 10: break
        print("  ", p.name)

    # Inspect 1 sample file
    sample = next(HUBERT_NEW.rglob("*.npy"))
    arr = np.load(sample)
    print("\n🧠 Sample embedding:")
    print("  File:", sample.name)
    print("  Shape:", arr.shape)
    print("  Dtype:", arr.dtype)
else:
    print("\n⚠️ No .npy files found yet — extraction might not have saved anything.")

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/NLI_HuBERT_Project")
MANIFEST = BASE / "metadata.csv"
EMB_DIR = BASE / "features" / "hubert_npy"

df = pd.read_csv(MANIFEST)
utt_col = next((c for c in ["wav_path","utt_id","path","filename"] if c in df.columns), None)
missing = []

for p in df[utt_col]:
    stem = Path(p).stem
    if not (EMB_DIR / f"{stem}.npy").exists():
        missing.append(stem)

print(f"✅ Checked {len(df)} entries")
print(f"🟢 Found all: {len(missing)==0}")
if missing:
    print(f"⚠️ Missing {len(missing)} embeddings (e.g., {missing[:5]})")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!find /content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy -type f -name "*.npy" | wc -l

In [ ]:
import os, numpy as np
from collections import Counter

folder = "/content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy"

files = [f for f in os.listdir(folder) if f.endswith('.npy')]
print("Total files:", len(files))

shapes = Counter()
nan_count = 0
example = None
for i,f in enumerate(files):
    arr = np.load(os.path.join(folder,f))
    shapes[arr.shape] += 1
    if np.isnan(arr).any():
        nan_count += 1
    if example is None:
        example = (f, arr.shape)
print("Unique shapes and counts:", shapes)
print("Files with NaNs:", nan_count)
print("Example file:", example)

In [ ]:
labels = [fn.split("_")[0] for fn in files]
from collections import Counter
Counter(labels)

In [ ]:
import numpy as np, os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

folder = "/content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy"
files = [f for f in os.listdir(folder) if f.endswith('.npy')]

X, y = [], []
for fn in files:
    arr = np.load(os.path.join(folder, fn))   # shape: (frames, dim) or (dim,)
    # If arr has frames, do mean pooling:
    if arr.ndim == 2:
        feat = arr.mean(axis=0)
    else:
        feat = arr
    X.append(feat)
    # adapt this to your filename -> label scheme:
    label = fn.split("_")[0]
    y.append(label)

X = np.vstack(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
import os
from collections import Counter

folder = "/content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy"
files = [f for f in os.listdir(folder) if f.endswith('.npy')]

# robust label extractor (handles "Andhra_speaker (64).npy")
def extract_label(fn):
    base = os.path.splitext(fn)[0]
    if base.endswith(')') and ' (' in base:
        return base.rsplit(' (', 1)[0]
    return base

labels = [extract_label(f) for f in files]
counts = Counter(labels)
print("Total files:", len(files))
print("Total classes:", len(counts))
print("Top 20 classes:", counts.most_common(20))
print("Classes with 1 sample:", [k for k,v in counts.items() if v == 1])
print("Classes with <5 samples:", [k for k,v in counts.items() if v < 5])

In [ ]:
# Drop singletons, train RandomForest baseline
import os, joblib, numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

folder = "/content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy"
files = [f for f in os.listdir(folder) if f.endswith('.npy')]

def extract_label(fn):
    base = os.path.splitext(fn)[0]
    if base.endswith(')') and ' (' in base:
        return base.rsplit(' (', 1)[0]
    return base

labels_all = [extract_label(f) for f in files]
counts = Counter(labels_all)
bad_labels = {lbl for lbl,c in counts.items() if c < 2}
print("Dropping classes (count <2):", len(bad_labels))
if bad_labels:
    print(sorted(list(bad_labels))[:30], "...")  # show up to 30 dropped labels

X, y = [], []
for fn in files:
    lbl = extract_label(fn)
    if lbl in bad_labels:
        continue
    arr = np.load(os.path.join(folder, fn))
    feat = arr.mean(axis=0) if arr.ndim == 2 else arr
    X.append(feat); y.append(lbl)

X = np.vstack(X); y = np.array(y)
print("Data after dropping singletons:", X.shape, "classes:", len(set(y)))

# encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

# stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)
print("Train/test shapes:", X_train.shape, X_test.shape)

# train RF
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
clf.fit(X_train, y_train)

# evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))

# save
out_dir = "/content/drive/MyDrive/NLI_HuBERT_Project/models"
os.makedirs(out_dir, exist_ok=True)
joblib.dump(clf, os.path.join(out_dir, "rf_hubert_baseline.joblib"))
joblib.dump(le, os.path.join(out_dir, "label_encoder.joblib"))
print("Saved model and label encoder to:", out_dir)

In [ ]:
# === Safe evaluation on files present in the saved LabelEncoder ===
import joblib, os, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score
import matplotlib.pyplot as plt

model_p = "/content/drive/MyDrive/NLI_HuBERT_Project/models/rf_hubert_baseline.joblib"
le_p    = "/content/drive/MyDrive/NLI_HuBERT_Project/models/label_encoder.joblib"
folder  = "/content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy"

clf = joblib.load(model_p)
le  = joblib.load(le_p)
known_labels = set(le.classes_.tolist())

files = [f for f in os.listdir(folder) if f.endswith('.npy')]
def extract_label(fn):
    base = os.path.splitext(fn)[0]
    if base.endswith(')') and ' (' in base:
        return base.rsplit(' (', 1)[0]
    return base

# Build dataset but only keep files whose label is known to the encoder
X, y, names = [], [], []
skipped = []
for fn in files:
    lbl = extract_label(fn)
    if lbl not in known_labels:
        skipped.append((fn, lbl))
        continue
    arr = np.load(os.path.join(folder, fn))
    feat = arr.mean(axis=0) if arr.ndim == 2 else arr
    X.append(feat); y.append(lbl); names.append(fn)

print(f"Total files found: {len(files)}")
print(f"Files kept for evaluation: {len(X)}")
print(f"Files skipped (unknown labels): {len(skipped)}")

if len(skipped) > 0:
    # show a sample of skipped labels
    from collections import Counter
    print("Sample skipped label counts:", Counter([s[1] for s in skipped]).most_common(10))

# if nothing left to evaluate, stop
if len(X) == 0:
    raise SystemExit("No files to evaluate after filtering. Retrain model or check label mapping.")

X = np.vstack(X)
y = np.array(y)

# encode y same as training
y_enc = le.transform(y)

# predict
y_pred_enc = clf.predict(X)
y_pred = le.inverse_transform(y_pred_enc)

# Metrics & report
acc = accuracy_score(y_enc, y_pred_enc)
print("Overall accuracy on filtered files:", acc)
print("\nClassification report:")
print(classification_report(y_enc, y_pred_enc, target_names=le.classes_))

# Confusion matrix (may be large)
cm = confusion_matrix(y_enc, y_pred_enc)
fig, ax = plt.subplots(figsize=(12,10))
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=ax, xticks_rotation=90, cmap='Blues', colorbar=True)
plt.title("Confusion Matrix (filtered classes)")
plt.show()

# Save filtered predictions to CSV
out_df = pd.DataFrame({"filename": names, "true_label": y, "pred_label": y_pred})
out_dir = "/content/drive/MyDrive/NLI_HuBERT_Project/results"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "rf_filtered_predictions.csv")
out_df.to_csv(out_path, index=False)
print("Saved filtered predictions to:", out_path)

# Top confused pairs
labels = list(le.classes_)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
pairs = []
for i in range(len(labels)):
    for j in range(len(labels)):
        if i!=j:
            pairs.append((labels[i], labels[j], cm_norm[i,j]))
pairs = sorted(pairs, key=lambda x: x[2], reverse=True)
print("\nTop 10 confused label pairs (true -> pred) with normalized rates:")
for a,b,r in pairs[:10]:
    print(f"{a} -> {b} : {r:.3f}")

In [ ]:
#  Retrain RF on HuBERT features while duplicating singleton classes once
import os, joblib, numpy as np
from collections import defaultdict, Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# paths
folder = "/content/drive/MyDrive/NLI_HuBERT_Project/features/hubert_npy"
out_dir = "/content/drive/MyDrive/NLI_HuBERT_Project/models"
os.makedirs(out_dir, exist_ok=True)

# helper to extract label from filename (handles "Label (idx).npy")
def extract_label(fn):
    base = os.path.splitext(fn)[0]
    if base.endswith(')') and ' (' in base:
        return base.rsplit(' (', 1)[0]
    return base

# load features grouped by label
files = [f for f in os.listdir(folder) if f.endswith('.npy')]
by_label = defaultdict(list)
for fn in files:
    lbl = extract_label(fn)
    arr = np.load(os.path.join(folder, fn))
    feat = arr.mean(axis=0) if arr.ndim == 2 else arr
    by_label[lbl].append(feat)

# duplicate singletons once (quick hack)
dup_count = 0
duplicated_labels = []
X_list, y_list = [], []
for lbl, feats in by_label.items():
    if len(feats) == 1:
        # duplicate once
        X_list.append(feats[0]); y_list.append(lbl)
        X_list.append(feats[0].copy()); y_list.append(lbl)
        dup_count += 1
        duplicated_labels.append(lbl)
    else:
        for f in feats:
            X_list.append(f); y_list.append(lbl)

X = np.vstack(X_list)
y = np.array(y_list)
print("Total files (after duplication):", X.shape[0])
print("Num classes:", len(set(y)))
print("Duplicated singleton classes:", dup_count)
if dup_count:
    print("Sample duplicated labels:", duplicated_labels[:10])

# encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Label classes:", len(le.classes_))

# stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

print("Train/test shapes:", X_train.shape, X_test.shape)

# train RF (balanced)
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
clf.fit(X_train, y_train)

# evaluate on test
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

# save model + label encoder
joblib.dump(clf, os.path.join(out_dir, "rf_hubert_retrained_dup_singletons.joblib"))
joblib.dump(le, os.path.join(out_dir, "label_encoder_retrained.joblib"))
print("Saved retrained model and label encoder to:", out_dir)

In [ ]:
# CNN on MFCC (2D conv treating MFCC as image)
# Paste this into Colab after mounting Drive (drive.mount('/content/drive'))
# Edit PATHS if needed, then run.

import os, math, random
from pathlib import Path
import numpy as np, pandas as pd
from collections import Counter
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import  DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import joblib

#  CONFIG
BASE = Path("/content/drive/MyDrive/NLI_HuBERT_Project")   # EDIT if different
MANIFEST = BASE/"metadata.csv"
MFCC_DIR = BASE/"features"/"mfcc"     # contains utt.npy (shape: n_mfcc x T) or (n_mfcc,) pooled
MODEL_DIR = BASE/"models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-3
N_MFCC = 40            # typical; we'll detect from first file if available
MAX_FRAMES = 300       # pad/truncate time dimension to this many frames
MIN_UTTS = 3           # drop very small classes


torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

#  Utilities
def find_mfcc_path(utt):
    # robust finder for utt stems - check common extensions
    for ext in (".npy", ".npz"):
        p = MFCC_DIR / f"{utt}{ext}"
        if p.exists(): return p
    # fallback: search files containing stem
    for p in MFCC_DIR.glob("*"):
        if utt in p.stem: return p
    return None

def load_mfcc_file(p):
    # returns numpy array shaped (n_mfcc, frames) OR 1D pooled vector
    if str(p).endswith(".npz"):
        d = np.load(p)
        # try to find MFCC array inside .npz
        arr = None
        for k in d.files:
            a = d[k]
            if a.ndim == 2:
                arr = a
                break
            if a.ndim == 1 and arr is None:
                arr = a
        if arr is None:
            raise RuntimeError("Couldn't parse npz: " + str(p))
        return arr
    else:
        arr = np.load(p)
        return arr

def pad_truncate(arr, max_frames=MAX_FRAMES):
    # arr shape: (n_mfcc, T) OR (n_features,) (pooled)
    if arr.ndim == 1:
        # pooled vector -> repeat to form small time axis (least preferred)
        vec = arr
        n_mfcc = min(len(vec), N_MFCC)
        # make 2D by tile
        out = np.zeros((n_mfcc, max_frames), dtype=np.float32)
        tile = vec[:n_mfcc]
        out[:n_mfcc, :] = np.tile(tile[:,None], (1, max_frames))
        return out
    n_mfcc, t = arr.shape
    if t >= max_frames:
        return arr[:, :max_frames]
    else:
        pad = np.zeros((n_mfcc, max_frames - t), dtype=arr.dtype)
        return np.concatenate([arr, pad], axis=1)

#   Build dataset index
df = pd.read_csv(MANIFEST)
# detect utt stem column robustly
candidate_utt_cols = ["utt_id","wav_path","filename","file","path","audio_path"]
utt_col = next((c for c in candidate_utt_cols if c in df.columns), candidate_utt_cols[1] if candidate_utt_cols[1] in df.columns else df.columns[0])

# Build list of available MFCCs
rows = []
missing = 0
for _, r in df.iterrows():
    utt_raw = str(r.get(utt_col, r.name))
    stem = Path(utt_raw).stem
    p = find_mfcc_path(stem)
    if p is None:
        missing += 1
        continue
    rows.append((stem, str(r.get("speaker_id", stem)), str(r.get("label","unknown")), str(p)))
print("Manifest rows:", len(df), "Found MFCC rows:", len(rows), "Missing:", missing)

if len(rows) == 0:
    raise RuntimeError("No MFCC files found. Check MFCC_DIR and manifest.")

# Filter small classes
labels_all = [r[2] for r in rows]
cnt = Counter(labels_all)
keep_labels = {lab for lab,c in cnt.items() if c >= MIN_UTTS}
print("Original classes:", len(cnt), "Keeping:", len(keep_labels))
rows = [r for r in rows if r[2] in keep_labels]
print("After filtering rows:", len(rows), "label counts:", Counter([r[2] for r in rows]))

#    Dataset class
class MFCCDataset(Dataset):
    def _init_(self, rows):
        self.rows = rows
    def _len_(self): return len(self.rows)
    def _getitem_(self, idx):
        stem, spk, lab, p = self.rows[idx]
        arr = load_mfcc_file(Path(p))
        arr2 = pad_truncate(arr, MAX_FRAMES)   # (n_mfcc, MAX_FRAMES)
        # detect n_mfcc dynamic; if necessary crop/pad freq axis to N_MFCC
        if arr2.shape[0] >= N_MFCC:
            arr2 = arr2[:N_MFCC, :]
        else:
            padfreq = np.zeros((N_MFCC - arr2.shape[0], arr2.shape[1]), dtype=arr2.dtype)
            arr2 = np.vstack([arr2, padfreq])
        # per-sample normalization (mean/std)
        arr2 = (arr2 - arr2.mean()) / (arr2.std() + 1e-9)
        # to tensor: shape (1, n_mfcc, frames)
        x = torch.tensor(arr2[None,:,:], dtype=torch.float32)
        return x, lab, spk, stem

#   prepare train/test splits (speaker-wise)
rows_arr = np.array(rows)
labels = [r[2] for r in rows]
groups = [r[1] for r in rows]

le = LabelEncoder().fit(labels)
y = le.transform(labels)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(rows_arr, y, groups))
train_rows = rows_arr[tr_idx].tolist()
test_rows  = rows_arr[te_idx].tolist()

print("Train rows:", len(train_rows), "Test rows:", len(test_rows))
print("Train label counts:", Counter([r[2] for r in train_rows]))
print("Test label counts:",  Counter([r[2] for r in test_rows]))
print("Train speakers:", len(set([r[1] for r in train_rows])), "Test speakers:", len(set([r[1] for r in test_rows])))

#   Dataloaders
train_ds = MFCCDataset(train_rows)
test_ds  = MFCCDataset(test_rows)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

#    Model (small 2D CNN)
class SmallCNN(nn.Module):
    def _init_(self, n_classes, in_ch=1):
        super()._init_()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, 32, kernel_size=(5,5), padding=2), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, kernel_size=(3,3), padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d((2,2)),
            nn.Conv2d(64,128,kernel_size=(3,3), padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes)
        )
    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

n_classes = len(le.classes_)
model = SmallCNN(n_classes).to(DEVICE)
print(model)

# Training utilities
optimizer = optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
best_val_acc = 0.0
patience = 4
patience_ctr = 0

def eval_model(dl):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, ylab, _, _ in dl:
            xb = xb.to(DEVICE)
            y_idx = le.transform(ylab)
            out = model(xb)
            p = out.argmax(dim=1).cpu().numpy()
            preds.extend(p.tolist())
            trues.extend(y_idx.tolist())
    acc = accuracy_score(trues, preds)
    return acc, preds, trues

#  Train loop
for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0
    for xb, ylab, _, _ in train_loader:
        xb = xb.to(DEVICE)
        y_idx = torch.tensor(le.transform(ylab), dtype=torch.long, device=DEVICE)
        out = model(xb)
        loss = criterion(out, y_idx)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    tr_loss = running_loss / len(train_loader.dataset)
    val_acc, _, _ = eval_model(test_loader)
    print(f"Epoch {epoch:02d} | train_loss: {tr_loss:.4f} | val_acc: {val_acc:.4f}")
    # early stop
    if val_acc > best_val_acc + 1e-4:
        best_val_acc = val_acc
        patience_ctr = 0
        # save best
        torch.save(model.state_dict(), MODEL_DIR/f"cnn_mfcc_best.pth")
        joblib.dump(le, MODEL_DIR/"label_encoder_mfcc.joblib")
        print(" Saved best model.")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print("Early stopping.")
            break

#   Final evaluation
model.load_state_dict(torch.load(MODEL_DIR/f"cnn_mfcc_best.pth", map_location=DEVICE))
val_acc, preds, trues = eval_model(test_loader)
print("Final test acc:", val_acc)
print(classification_report(trues, preds, target_names=le.classes_, zero_division=0))

# Confusion matrix
cm = confusion_matrix(trues, preds)
plt.figure(figsize=(8,6))
import seaborn as sns
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"MFCC CNN Confusion Matrix (acc={val_acc:.3f})")
plt.show()

print("Saved model & label encoder to", MODEL_DIR)  (partial)

In [ ]:
# View training results
import json
from IPython.display import Image, display

with open('results/dl_models_results.json', 'r') as f:
    results = json.load(f)

print("="*70)
print("GPU TRAINING RESULTS")
print("="*70)
print(f"Device Used: {results['device']}")
print(f"Training Samples: {results['num_samples']}")
print(f"Epochs: {results['epochs']}")
print("\n" + "-"*70)
print(f"{'Model':<20} {'Best Val Acc':<20}")
print("-"*70)
for model, metrics in results['models'].items():
    print(f"{model:<20} {metrics['best_val_acc']:>18.2f}%")
print("="*70)

# Display comparison plot
display(Image('results/dl_models_comparison.png'))

In [ ]:
# Comprehensive results from all experiments
import json

print("="*90)
print(" "*30 + "PROJECT RESULTS")
print("="*90)

# Collect all results
all_results = {}

# MFCC Baseline
try:
    with open('models/mfcc_baseline_info.json') as f:
        all_results['MFCC + RF (Baseline)'] = json.load(f)['accuracy'] * 100
except:
    pass

# HuBERT + RF
try:
    with open('models/hubert_training_info.json') as f:
        all_results['HuBERT Layer 3 + RF'] = json.load(f)['accuracy'] * 100
except:
    pass

# Layer-wise best
try:
    with open('results/hubert_layerwise_results.json') as f:
        layer_data = json.load(f)
        best_acc = max([r['accuracy'] for r in layer_data['layer_results']]) * 100
        all_results['HuBERT Best Layer + RF'] = best_acc
except:
    pass

# Deep Learning Models
try:
    with open('results/dl_models_results.json') as f:
        dl_data = json.load(f)
        for model_name, metrics in dl_data['models'].items():
            all_results[f'HuBERT + {model_name}'] = metrics['best_val_acc']
except:
    pass

# Display
print(f"\n{'Approach':<45} {'Validation Accuracy':<20}")
print("-"*65)
for approach in sorted(all_results.keys()):
    print(f"{approach:<45} {all_results[approach]:>18.2f}%")

if all_results:
    best_approach = max(all_results.items(), key=lambda x: x[1])
    print("\n" + "="*65)
    print(f"🏆 Best: {best_approach[0]}")
    print(f"🏆 Accuracy: {best_approach[1]:.2f}%")
    print("="*65)

print("\n✅ All experiments completed!")
print("="*90)

---
## Complete Results Summary

Final comparison of all approaches:

### Model Architectures (Same as CPU version)

All three architectures automatically utilize GPU when available:

1. **CNN**: 3 Conv1D layers with batch norm and pooling
2. **BiLSTM**: 2-layer bidirectional LSTM (256 hidden units)
3. **Transformer**: 4 layers, 8 heads, positional encoding

**GPU Optimizations:**
- Automatic CUDA device placement
- Batch processing (32 samples)
- Efficient memory management
- Fast matrix operations on GPU

Expected speedup: **10-50x faster** than CPU depending on GPU model.

In [ ]:
# Run GPU-accelerated training
# The script automatically detects and uses GPU if available

!python scripts/train_dl_models.py

---
## GPU-Accelerated Deep Learning Training

Training CNN, BiLSTM, and Transformer models with GPU acceleration.

---

## GPU Training Implementation

Due to notebook execution challenges, the production GPU training was moved to `scripts/` and executed via terminal.

### Key Script
- `scripts/train_final_model.py` - Optimized GPU training

### Final Production Model
- **Architecture**: Random Forest (300 estimators)
- **Features**: HuBERT Layer 3 (768-dim → PCA 128-dim)
- **Test Accuracy**: 99.73%
- **F1-Score**: 99.69%

### Feature Re-extraction
All features were re-extracted using persistent HuBERT model to ensure consistency:
```bash
python scripts/batch_extract_features.py
```

### Model Artifacts
- `models/rf_hubert_final.joblib` - Trained classifier
- `models/scaler_hubert.joblib` - Feature scaler
- `models/pca_hubert.joblib` - PCA reducer
- `models/label_encoder.joblib` - Label encoder
- `models/class_stats.npz` - Open-set detection statistics

### Usage
```python
from scripts.predict_backend import predict_from_path

result = predict_from_path("audio.wav")
print(f"Predicted: {result['predicted_label']} ({result['confidence']:.2%})")
```